# VoxBridge - Voice Clone Proof (free GPU)

This proves the core of the dubbing idea: take a sample of someone's voice and make **that same voice** speak another language.

It runs here because Colab gives you a **free GPU + ~12GB RAM** - the resources an 8GB no-GPU laptop does not have.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU**.

Then Runtime -> Run all, and upload a voice sample when asked (a 6-15s WAV of one person talking - e.g. the `reference_voice.wav` from your project, or any clip).

In [ ]:
# 1. Install the voice-cloning stack (~2-3 min)
!pip -q install "coqui-tts" "transformers>=4.57,<5.0" 2>/dev/null
import torch
print('GPU available:', torch.cuda.is_available())

In [ ]:
# 2. Upload a reference voice (one person, ~6-15s, WAV)
from google.colab import files
print('Choose a voice sample to clone...')
uploaded = files.upload()
ref_path = list(uploaded.keys())[0]
print('Reference:', ref_path)

In [ ]:
# 3. Load XTTS v2 (downloads ~1.8GB the first time)
import os
os.environ['COQUI_TOS_AGREED'] = '1'
from TTS.api import TTS
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda' if torch.cuda.is_available() else 'cpu')
print('Model ready.')

In [ ]:
# 4. Make the SAME voice speak Hindi and English
from IPython.display import Audio, display

jobs = {
    'hindi.wav':   ('hi', 'मैंने इस गाँव में सत्तर साल बिताए हैं, और मैंने कई सर्दियाँ आते-जाते देखी हैं।'),
    'english.wav': ('en', 'I have lived in this village for seventy years, and I have seen many winters come and go.'),
}

print('--- ORIGINAL reference voice ---')
display(Audio(ref_path))

for out, (lang, text) in jobs.items():
    tts.tts_to_file(text=text, speaker_wav=ref_path, language=lang, file_path=out)
    print(f'--- SAME voice, language={lang} ---')
    display(Audio(out))

print('\nCompare: does the cloned voice sound like the same person? That is the whole test.')